# NavLoRI-Fusion — Run-2 Walkthrough

Paper-facing walkthrough of the run-2 archive (2026-05-25 → 2026-05-26).
Source-of-truth narrative: [`handoff/SUMMARY.md`](../handoff/SUMMARY.md).

**Exclusions** per [`handoff/SCIENTIST_NOTE_notebook-exclusions.md`](../handoff/SCIENTIST_NOTE_notebook-exclusions.md):
- IPIN 2024 floor 0 dropped from paper-facing rows (RESULT_22 β5).
- MoTTransformer dropped from paper-facing arch columns (RESULT_21 γ5).
Both stay in the repo for reproducibility; this notebook covers 6 datasets × 3 architectures.

**Verdict**: `GOAL_REACHED: true with documented limitations`. Limitations are part of the contribution — they delineate where fusion helps, where it saturates, and what the open lever is.

Author: Mohamed Bachar (CESI LINEACT). Target venue: PerCom 2026.

## Sections
- §0 Datasets pre-section (6 datasets × stats / overview / preprocessing demos)
- §1 Phase A: encoder audit (Anchor2Vec / IMUCNN / DPVOMotion / OdomCNN)
- §2 Phase B: 3-architecture fusion bake-off (incumbent / **CNN1D winner** / LSTM-attn runner-up)
- §3 Phase C: cross-dataset main results table (6 rows × 9 columns)
- §4 Honest gaps (smoothness debt, C2 raw-ATE, IMUWiFine test-no-IMU, TartanAir paper-soft)
- §5 Reproducibility (setup + per-script commands)

In [1]:
# Import + working-dir setup. Run this once.
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

ROOT = Path('.').resolve()
while ROOT.name != 'navlori-fusion' and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.pipeline.data import list_datasets, dataset_stats, preprocessing_demo
from src.pipeline.visualization import (
    plot_dataset_overview, plot_preprocessing_demo,
    plot_subset_eval_bar, plot_staleness_curve, plot_main_results_heatmap,
    set_paper_style,
)
set_paper_style()
print(f'ROOT = {ROOT}')
print(f'datasets = {list_datasets()}')

ROOT = X:\navlori-fusion
datasets = ['webots', 'msiln_site1_b1', 'imuwifine_floor4', 'ipin2024_floor0', 'ronin_canonical', 'tartanair_hospital', 'uji_indoorloc']


## §0 — Datasets pre-section

For each of the 6 paper-facing datasets, we show: (a) `dataset_stats(name)` summary, (b) `plot_dataset_overview(name)` multi-panel figure, (c) `preprocessing_demo(name, modality)` for the dataset's primary modality.

Each dataset's `stats()['known_caveats']` surfaces honest run-2 findings inline.

### §0.1 — Webots (4-modality, the only dataset with all 4)

Custom collection (TIAGO++ in Webots R2025a). 18 paths. **Phase B winner CNN1D test 0.339 m here.**

In [2]:
stats = dataset_stats('webots')
for k, v in stats.items():
    if k == 'known_caveats':
        print(f'  known_caveats:')
        for c in v: print(f'    - {c}')
    else:
        print(f'  {k}: {v}')

  name: webots
  collection_dir: data\async_collection
  modalities_available: ['wifi', 'imu', 'camera', 'odom']
  splits: {'train': [1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12], 'val': [2, 13, 14], 'test': [15, 16, 17]}
  sensor_rates_hz: {'imu': 31, 'odom': 15, 'gt': 10, 'wifi': 1, 'camera': 5}
  n_paths_total: 18
  n_paths: 18
  duration_total_s: 0.0
  duration_per_path_mean: 0.0
  duration_per_path_min: 0.0
  duration_per_path_max: 0.0
  known_caveats:
    - Path 0 is empty / failed-collection; excluded.
    - Webots WiFi is GPR-synthesised, not measured — sub-metre MAE optimistic vs real-world (CLAUDE.md honest finding #3).
  source_result: RESULT_06+ (fusion); RESULT_03/04 (per-leg encoder audits)


In [3]:
fig = plot_dataset_overview('webots')
plt.show()

In [4]:
# Webots representative preprocessing demo: WiFi (others in stats() are imu/camera/odom).
demo = preprocessing_demo('webots', 'wifi')
print(demo['description_raw'])
print('->', demo['description_preprocessed'])
fig = plot_preprocessing_demo(demo, 'wifi')
plt.show()

raw RSSI dBm in [-100, 0]; NaN = AP not visible (n=1 scan(s), 117 APs)
-> NaN -> -100 (no signal), then affine (x+100)/100 to [0, 1]


### §0.2 — IMUWiFine floor 4 (WiFi+IMU, real-world; RESULT_19/20)

In [5]:
stats = dataset_stats('imuwifine_floor4')
print(f"modalities: {stats['modalities_available']}")
print(f"n_paths: {stats.get('n_paths_total', '?')}")
print(f"\nknown caveats:")
for c in stats['known_caveats']: print(f'  - {c}')

modalities: ['wifi', 'imu']
n_paths: 80

known caveats:
  - **Test paths lack IMU by dataset design** (RESULT_20 audit). Fusion test = WiFi-only inference.
  - Two raw formats: train+val Android logger (WiFi 0.31 Hz, IMU 30 Hz); test header-less (WiFi 5.65-6.57 Hz, no IMU).
  - Test physical region constrained to y=1.2-1.6 m vs train+val 0-5 m (cross-session, separate campaign).
  - Val/test gap +408 % on CNN1D is failure mode 3 (legitimate cross-session shift, not code bug).


In [6]:
fig = plot_dataset_overview('imuwifine_floor4')
plt.show()

### §0.3 — MSILN site1/B1 (cross-session real-world; RESULT_15)

In [7]:
stats = dataset_stats('msiln_site1_b1')
print(f"modalities: {stats['modalities_available']}")
print(f"n_paths: {stats.get('n_paths_total', '?')}")
print(f"\nknown caveats:")
for c in stats['known_caveats']: print(f'  - {c}')

modalities: ['wifi', 'imu']
n_paths: 133

known caveats:
  - Cross-session train (Nov 24) / val (Nov 25) / test (Dec 5+6).
  - Test path 130 (786 samples, ~28 % of test, WiFi-dense) dominates kNN test mean — RESULT_15.
  - WiFi RSSI fingerprints drift across sessions; gate (c)-1 partial only.


In [8]:
fig = plot_dataset_overview('msiln_site1_b1')
plt.show()

### §0.4 — RoNIN canonical (IMU only, unseen-subjects benchmark; RESULT_07/23)

In [9]:
stats = dataset_stats('ronin_canonical')
print(f"modalities: {stats['modalities_available']}")
print(f"canonical unseen present: {stats['n_unseen_present_locally']}/{stats['n_unseen_canonical']}")
print(f"canonical train present: {stats['n_train_present_locally']}/{stats['n_train_canonical']}")
print(f"\nknown caveats:")
for c in stats['known_caveats']: print(f'  - {c}')

modalities: ['imu']
canonical unseen present: 32/32
canonical train present: 69/73

known caveats:
  - Window=200 RoNIN convention; our IMUCNN window is 32 (RESULT_23 chunks 200 into K=4 sub-windows of 50).
  - ATE/RTE/Umeyama via vendored RoNIN metric.compute_ate_rte (NEVER hand-rolled SVD per amended rubric correction #3).
  - Pretrained ResNet1D reproduces paper's 5.140 m exactly (RESULT_07).
  - C2 audit: IMUCNN canonical raw 9.961 m / Umeyama 7.876 m vs ResNet1D 5.14 m — raw +94 % outside 20 % gate (RESULT_07).
  - CNN1D aggregator over IMUCNN sub-windows: raw 7.59 m / Umeyama 5.95 m — Umeyama gate cleared at +15.7 % (RESULT_23).


### §0.5 — TartanAir hospital P000 (Camera only; RESULT_08)

In [10]:
stats = dataset_stats('tartanair_hospital')
print(f"modalities: {stats['modalities_available']}")
print(f"n_frames: {stats.get('n_frames', '?')}")
print(f"split convention: {stats['split_convention']}")
print(f"\nknown caveats:")
for c in stats['known_caveats']: print(f'  - {c}')

modalities: ['camera']
n_frames: 0
split convention: first 80 % train / last 20 % test slice (RESULT_08)

known caveats:
  - Image-only TartanAir v1 — NO IMU; only Camera modality.
  - RESULT_08 reports TartanVO last-20 % ATE 0.012 m vs DPVOMotion 0.293 m → +2300 % gap, paper-soft per-leg verdict.
  - Used to validate the DPVOMotion encoder's trunk transferability (Mode α Webots-trained head infeasible without saved head).


### §0.6 — UJIIndoorLoc (WiFi only, per-scan; RESULT_01/24)

In [11]:
stats = dataset_stats('uji_indoorloc')
print(f"modalities: {stats['modalities_available']}")
print(f"splits: {stats['splits']}")
print(f"n_aps: {stats['n_aps']}")
print(f"\nknown caveats:")
for c in stats['known_caveats']: print(f'  - {c}')

modalities: ['wifi']
splits: {'train': 19937, 'validation': 1111}
n_aps: 520

known caveats:
  - Per-scan data — NO temporal axis. K=1 + M=1 degenerate row of the main-results table (RESULT_24 α7).
  - wlan_localization global SOTA val mean Euclidean 15.17 m; Anchor2Vec 8.69 m (RESULT_01); CNN1D 8.72, LSTM-attn 8.43 (RESULT_24).
  - No test split — `validationData.csv` is the benchmark. Per-scan distribution reported instead of per-trajectory smoothness.


## §1 — Phase A: encoder audit

Per-modality encoder audit. Each subsection: load the encoder + call `demo_forward(sample)` → introspect raw / intermediate / encoded; compare against published SOTA on the canonical benchmark.

### §1.1 — WiFi: Anchor2Vec on UJIIndoorLoc → **keep** (RESULT_01)

- wlan_localization (SOTA, global mode): **15.17 m** val mean Euclidean.
- Anchor2Vec (ours, 0.075 M params):      **8.69 m** val (−43 % vs SOTA).

Audit verdict: **keep**. Anchor2Vec beats the SOTA by 43 % at one-quarter the param budget.

In [12]:
from src.pipeline.encoders import Anchor2Vec
from src.pipeline.data import load_dataset

Xva, _ = load_dataset('uji_indoorloc', split='validation')
raw = np.where(Xva[:1] == 100, -100.0, Xva[:1]).clip(-100, 0)
raw = (raw + 100.0) / 100.0
enc = Anchor2Vec(n_aps=Xva.shape[1], embed_dim=128, n_anchors=64)
demo = enc.demo_forward(raw)
print(demo['description'])
print(f"  raw shape:          {demo['raw'].shape}")
print(f"  intermediate shape: {demo['intermediate'].shape}  (anchor attention weights)")
print(f"  encoded shape:      {demo['encoded'].shape}")

Anchor2Vec: 520 APs -> 64 learned anchors (softmax-attention weights are the intermediate) -> 128-d token via weighted sum of anchor embeddings + MLP head.
  raw shape:          (1, 520)
  intermediate shape: (1, 64)  (anchor attention weights)
  encoded shape:      (1, 128)


### §1.2 — IMU: IMUCNN on RoNIN canonical → **keep (in-domain only)** (RESULT_07)

- RoNIN ResNet1D (SOTA, pretrained, 4.24 M): **5.140 m** raw ATE on canonical unseen — **paper-exact reproduction** (0.0 % delta).
- IMUCNN (ours, 0.05 M):                       9.961 m raw ATE / 7.876 m Umeyama.

Audit verdict: **`keep (in-domain only)`**. Canonical raw gap +94 % outside the 20 % gate; Umeyama +53 % also outside. Per amended-rubric correction #3 (raw weighted ≥ aligned), C2 not discharged.

In [13]:
from src.pipeline.encoders import IMUCNN
imu_enc = IMUCNN(in_features=6, embed_dim=128)
# synthetic 32-step 6-ch IMU window
x = np.random.randn(1, 32, 6).astype(np.float32) * 0.5
demo = imu_enc.demo_forward(x)
print(demo['description'])
print(f"  raw shape:          {demo['raw'].shape}")
print(f"  intermediate shape: {demo['intermediate'].shape}  (conv stack pre-pooling activations)")
print(f"  encoded shape:      {demo['encoded'].shape}")

IMUCNN: 1D-CNN (6->128 ch) over window=32 (6-channel input) -> global avg pool -> 128-d token.
  raw shape:          (1, 32, 6)
  intermediate shape: (1, 128, 32)  (conv stack pre-pooling activations)
  encoded shape:      (1, 128)


### §1.3 — Camera: DPVOMotionEncoder on TartanAir hospital → **keep with smoothness debt** (RESULT_03/08)

- TartanVO (SOTA, full SLAM): **0.518 m** full-sequence / **0.012 m** last-20 % slice (Umeyama-aligned ATE).
- DPVOMotion (ours, Mode α):                                0.293 m last-20 % slice.

Audit verdict: **paper-soft**. Last-20 % gap +2300 % vs TartanVO; per-sample MAE on Webots is fit-for-purpose as a *fusion encoder* (anchor = WiFi), not a standalone VO baseline. Smoothness debt r ≈ 0.07 documented (RESULT_03).

`DPVOMotionEncoder.demo_forward(image_pair)` shape: (B, n_patches=64, 132) = trunk_feat (128) + dx + dy + ‖flow‖ + corr_peak. We skip loading the full encoder here to avoid the DPVO weights dependency; the demo runs in §0.5 dataset overview.

### §1.4 — Odom: OdomCNN on Webots → **keep** (RESULT_04)

- Trivial cumulative-integration floor: 8.27 m test (high MAE, perfect smoothness).
- OdomCNN-P-B (ours, 0.015 M, Δ-features): **4.24 m** test (−49 % vs floor; smoothness r ≈ 0).

Audit verdict: **keep**. P-B Δ-features preprocessing beats P-A raw normalisation by 49 %. Honest weakness: smoothness r ≈ 0 vs trivial-floor r=0.999 → Phase B feeds both OdomCNN embedding (absolute-MAE) and raw integrated `(odom_x, odom_y)` (smoothness).

In [14]:
from src.pipeline.encoders import OdomCNN
odom_enc = OdomCNN(in_features=7, embed_dim=128)
x = np.random.randn(1, 16, 7).astype(np.float32) * 0.3
demo = odom_enc.demo_forward(x)
print(demo['description'])
print(f"  raw shape:          {demo['raw'].shape}")
print(f"  intermediate shape: {demo['intermediate'].shape}  (conv stack pre-pooling)")
print(f"  encoded shape:      {demo['encoded'].shape}")

OdomCNN: 1D-CNN over window=16 (7-channel input) -> 64 channels -> 128-d token. P-B Δ-features (RESULT_04 winner) recommended at the dataset stage.
  raw shape:          (1, 16, 7)
  intermediate shape: (1, 64, 16)  (conv stack pre-pooling)
  encoded shape:      (1, 128)


## §2 — Phase B: 3-architecture fusion bake-off (Webots K=4 4-modality)

Run-2 final: 3 architectures benchmarked at the same K=4 + 4-mod + B=128 + lr=1.3e-3 protocol on full Webots data.

(MoTTransformer's outcome γ5 negative result kept in repo for reproducibility — `src/pipeline/fusion/mot_transformer.py` — but excluded from paper-facing presentation per `handoff/SCIENTIST_NOTE_notebook-exclusions.md`.)

In [15]:
# Phase B headline comparison (numbers cited from RESULT_13/14/17/18; load_trained available
# for live reproduction in cells below).
phase_b = pd.DataFrame([
    {'arch': 'incumbent (run-1)', 'params_M': 1.55, 'val_MAE': 0.394, 'test_MAE': 0.417,
     'smoothness_r': 0.039, 'latency_b1_ms': 6.41, 'source': 'RESULT_13/14'},
    {'arch': 'CNN1D (winner)',    'params_M': 0.51, 'val_MAE': 0.282, 'test_MAE': 0.339,
     'smoothness_r': 0.009, 'latency_b1_ms': 4.73, 'source': 'RESULT_17/18'},
    {'arch': 'LSTM-attn',         'params_M': 0.57, 'val_MAE': 0.301, 'test_MAE': 0.340,
     'smoothness_r': 0.051, 'latency_b1_ms': 4.67, 'source': 'RESULT_17/18'},
])
phase_b

,arch,params_M,val_MAE,test_MAE,smoothness_r,latency_b1_ms,source
0,incumbent (run-1),1.55,0.394,0.417,0.039,6.41,RESULT_13/14
1,CNN1D (winner),0.51,0.282,0.339,0.009,4.73,RESULT_17/18
2,LSTM-attn,0.57,0.301,0.340,0.051,4.67,RESULT_17/18


### Subset eval on the CNN1D winner (16 rows from RESULT_18)

Reveals the **cooperative-fusion regime**: WiFi anchors absolute position; motion modalities contribute marginal corrections. `drop:Odom` (= `wifi+imu+camera`) marginally beats full (test 0.338 vs full 0.339) — Odom is at the noise margin.

In [16]:
# Cached subset eval from RESULT_17 (see runs/overnight/run2_iter_17/cnn1d/.../all_subsets_test.json).
import json
ckpt_dir = next(Path('runs/overnight/run2_iter_17/cnn1d').glob('fusion_*'))
with open(ckpt_dir / 'all_subsets_test.json') as f:
    subs = json.load(f)
subset_dict = {k: v['mae'] for k, v in subs.items()}
fig = plot_subset_eval_bar(subset_dict, title='CNN1D Webots test — 16-row subset eval (RESULT_18)')
plt.show()

### WiFi staleness sweep on CNN1D (8-lag, RESULT_14 paper figure)

Linear slope **0.028 m/s, R²=0.995**: temporal fusion converts a staleness cliff into a gentle slope. Architecture-invariant property (incumbent: 0.029 m/s; LSTM-attn: 0.024).

In [17]:
# Cached staleness from RESULT_18 ablations JSON (see runs/overnight/run2_iter_18/cnn1d_ablations.json).
stale_p = Path('runs/overnight/run2_iter_18/cnn1d_ablations.json')
if stale_p.is_file():
    with open(stale_p) as f: a = json.load(f)
    lags = sorted(int(k) for k in a['staleness'].keys())
    mae = [a['staleness'][str(l)] for l in lags]
    slope = a['staleness_slope']['slope_m_per_s']
    fig = plot_staleness_curve(lags, mae, label='cnn1d', slope=slope,
                               title='CNN1D WiFi staleness — RESULT_18')
    plt.show()
else:
    print(f'Cached ablation JSON not present at {stale_p}; run scripts/_iter18_cnn1d_ablations.py to regenerate.')

### LSTM-attn dead-reckoning regime (structural finding, 3 datasets × 4 scenarios)

On the LSTM-attn checkpoint, all 4 `only:X` subsets are within ~8 % of full-fusion — the architecture learns to dead-reckon from any single modality, not just the WiFi anchor. The same regime replicates on IMUWiFine (`only:imu` 1.263 ≈ full 1.264) and IPIN floor 0 (`only:imu` 22.64 ≈ full 22.45).

Paper-grade discussion finding: **three distinct fusion regimes** emerge from the bake-off (CNN1D cooperative / LSTM-attn dead-reckoning / MoTTransformer WiFi-anchored).

In [18]:
ckpt_dir_lstm = next(Path('runs/overnight/run2_iter_17/lstm_attn').glob('fusion_*'))
with open(ckpt_dir_lstm / 'all_subsets_test.json') as f:
    subs_lstm = json.load(f)
subset_dict_lstm = {k: v['mae'] for k, v in subs_lstm.items()}
fig = plot_subset_eval_bar(subset_dict_lstm,
                            title='LSTM-attn Webots test — only:X ≈ full (dead-reckoning regime, RESULT_18)')
plt.show()

## §3 — Phase C: cross-dataset main results table

The paper-ready 6-row × 9-column comparison. Exclusions in effect (no IPIN row, no MoTTransformer column).

Source-of-truth: `handoff/SUMMARY.md`. Live render via `MainResultsTable.from_archive()`.

In [19]:
from src.pipeline.evaluation import MainResultsTable
table = MainResultsTable.from_archive()
df = table.to_dataframe()
df

,dataset,wlan_localization,RoNIN_ResNet1D,TartanVO,Anchor2Vec,DPVOMotion,IMUCNN,incumbent,cnn1d,lstm_attn
0,webots,n/a,n/a,n/a,n/a,n/a,n/a,0.39 v / 0.42 t,0.28 v / 0.34 t,0.30 v / 0.34 t
1,imuwifine_floor4,4.17 v / 8.50 t,26.84 v,n/a,n/a,n/a,n/a,n/a,1.40 v / 7.09 t,1.26 v / 7.20 t
2,msiln_site1_b1,21.26 v / 28.31 t,n/a,n/a,n/a,n/a,n/a,16.60 v / 14.02 t,n/a,n/a
3,ronin_canonical,n/a,5.14 t,n/a,n/a,n/a,9.96 t,n/a,7.59 t,7.50 t
4,tartanair_hospital,n/a,n/a,0.01 t,n/a,0.29 t,n/a,n/a,n/a,n/a
5,uji_indoorloc,15.17 v,n/a,n/a,8.69 v,n/a,n/a,n/a,8.72 v,8.43 v


### Per-leg validation status (criterion (a))

- **WiFi** (UJI): Anchor2Vec **8.69 m** vs wlanloc 15.17 m → ✅ beats by 43 %.
- **IMU** (RoNIN canonical): IMUCNN 9.96 m raw vs ResNet1D 5.14 m → ⚠ partial (Umeyama 7.88 within gate; raw outside). C2 labelled `keep (in-domain only)` per amended-rubric correction #3.
- **Camera** (TartanAir hospital): DPVOMotion 0.293 m vs TartanVO 0.012 m last-20 % → ⚠ paper-soft.
- **Odom** (Webots): internal, no public SOTA. OdomCNN-P-B keep verdict.

### C3 fusion claim (criterion (b))

**CNN1D test 0.339 m on Webots 4-modality K=4** clears the 0.5 m gate by **32 %**.

### C4 cross-session (criterion (c))

MSILN site1/B1 (PLAN_15 deployed config): val 16.60 / test 14.02. Beats wlan_localization SOTA by +4.66 m val / +14.29 m test (gate (c)-2 ✅ cleanly). Beats WiFi-kNN by only 1.06 m val / regresses on test (path-130 composition; gate (c)-1 partial). PLAN_25b candidate: re-run with CNN1D + Anchor2Vec.

## §4 — Honest gaps (documented limitations)

Run-2 surfaced four documented limitations. Each is part of the contribution (delineates where fusion helps, where it saturates, what the open lever is) — not a failure.

### §4.1 Smoothness debt is architecture-invariant

Per-trajectory smoothness median Pearson r ≤ 0.10 across **4 architectures × 5+ datasets**. The locked r > 0.20 gate (criterion (d)) is **never cleared**. The architectural-lever-for-smoothness hypothesis is **falsified**; the open lever is the loss function (auxiliary velocity B-1 / EMA token smoothing B-2 from RESULT_05).

In [20]:
smoothness = pd.DataFrame([
    {'arch': 'incumbent',      'Webots': 0.039, 'IMUWiFine': None,  'IPIN': None,  'MSILN': 0.107, 'source': 'RESULT_14/15'},
    {'arch': 'CNN1D (winner)', 'Webots': 0.009, 'IMUWiFine': -0.005,'IPIN': 0.067, 'MSILN': None,  'source': 'RESULT_18/19/22'},
    {'arch': 'LSTM-attn',      'Webots': 0.051, 'IMUWiFine': -0.007,'IPIN': 0.089, 'MSILN': None,  'source': 'RESULT_18/19/22'},
])
smoothness.set_index('arch')

,Webots,IMUWiFine,IPIN,MSILN,source
arch,,,,,
incumbent,0.039,NaN,NaN,0.107,RESULT_14/15
CNN1D (winner),0.009,-0.005,0.067,NaN,RESULT_18/19/22
LSTM-attn,0.051,-0.007,0.089,NaN,RESULT_18/19/22


### §4.2 C2 raw-ATE gap on canonical RoNIN

IMUCNN canonical raw ATE 9.96 m vs ResNet1D 5.14 m = +94 % outside the 20 % audit gate. Aggregator extension (CNN1D over K=4 IMUCNN sub-windows) improves to **7.59 m raw / 5.95 m Umeyama** (RESULT_23): **CNN1D's Umeyama gap +15.7 % clears the 20 % gate**, raw +47 % does not. Per amended-rubric correction #3 (raw wins), C2 labelled `keep (in-domain only)`.

### §4.3 IMUWiFine test = WiFi-only (dataset design, RESULT_20)

IMUWiFine test paths carry no IMU; fusion test column on this row is structurally WiFi-only inference. Failure mode 3 — legitimate cross-session/cross-campaign shift (wlanloc shows the same +104 % val/test gap, ruling out our-code as the culprit).

### §4.4 TartanAir paper-soft per-leg (RESULT_08)

DPVOMotion uses a Webots-trained head (Mode α infeasible without saved head; in-domain linear probe on first-80 % P000). +2300 % gap to TartanVO on last-20 % slice. The encoder is fit-for-purpose as a *fusion encoder* (WiFi anchors); not a standalone VO baseline.

## §5 — Reproducibility

### Setup

```bash
git clone https://github.com/<owner>/navlori-fusion.git
cd navlori-fusion
git submodule update --init --recursive   # pulls external_methods/{wlan_localization,ronin,tartanvo,dpvo}
python -m venv .venv
.venv\Scripts\activate                     # PowerShell
pip install -e ".[dev]"
```

See [`docs/EXTERNAL_DEPENDENCIES.md`](../docs/EXTERNAL_DEPENDENCIES.md) for the per-submodule setup notes (TartanVO weights, RoNIN pretrained ResNet1D, DPVO Windows-build limitation).

### Per-cell reproduction commands

| number | command | RESULT |
|--------|---------|--------|
| wlanloc 15.17 m + Anchor2Vec 8.69 m on UJI | `python scripts/eval_uji.py` | RESULT_01 |
| ResNet1D 5.140 m on canonical RoNIN unseen | `python scripts/eval_ronin_canonical.py` | RESULT_07 |
| TartanVO 0.012 m + DPVOMotion 0.293 m on TartanAir hospital | `python scripts/eval_tartanair_hospital.py` | RESULT_08 |
| wlanloc on IMUWiFine fl.4 (val 4.17 / test 8.50) | `python scripts/_eval_wlanloc_imuwifine.py` | RESULT_19 |
| wlanloc on MSILN cross-session (val 21.26 / test 28.31) | `python scripts/_eval_wlanloc_msiln.py` | RESULT_15 |
| CNN1D Webots 0.339 m (full retrain) | `python scripts/_train_webots_4mod_arch.py --arch cnn1d` | RESULT_17 |
| CNN1D ablation suite | `python scripts/_iter18_cnn1d_ablations.py --arch cnn1d --lags full` | RESULT_18 |

Archive paths:
- `handoff/results/RESULT_01-30.md` — per-iteration findings.
- `handoff/SUMMARY.md` — run-2 archive one-pager.
- `runs/overnight/run2_iter_*/` — saved checkpoints, JSONs, per-trajectory plots.
- `src/pipeline/{baselines,data,fusion,training,evaluation,visualization}/` — consolidated APIs (PLAN_26-29).

### 4-architecture bake-off note

`src/pipeline/fusion/mot_transformer.py` is a documented honest-negative experiment (RESULT_21 γ5: worst of 4 architectures on Webots). Kept in the repo for reproducibility per the run-2 methodology; excluded from the paper-facing presentation per `handoff/SCIENTIST_NOTE_notebook-exclusions.md`.